# 00 · Instalando o MongoDB no seu computador

**Objetivo:** sair deste notebook com um servidor MongoDB rodando no seu computador e o Python conectado a ele.

**Você vai precisar de:** Windows 10 ou 11, VS Code com este notebook aberto e cerca de 15 minutos. Quase tudo é feito pelo **terminal do VS Code**.

### Roteiro das aulas

| Notebook | Etapa | Exercícios |
|---|---|---|
| **00** | Instalar o MongoDB no seu computador e testar a conexão | — |
| **01** | JSON e CRUD | 1 |
| **02** | Consultas, arrays, agregação e paginação | 2 |
| **03** | Conexão robusta e primeira aplicação web (Streamlit) | 3 e 4 |
| **04** | Banco de imagens com GridFS (faces da FEI) | 5, 6 e 7 |
| **05** | Galeria de faces (app Streamlit) + desafio | 8 e desafio |


### Etapas deste notebook
1. Abrir o terminal
2. Conferir o `winget`
3. Instalar o MongoDB Server
4. Conferir se o MongoDB está rodando
5. Criar o arquivo `.env`
6. Instalar as bibliotecas e testar a conexão
7. Criar o primeiro banco
8. Problemas comuns

---
## O que vamos montar

```
 Seu notebook (VS Code)  ── mongodb://localhost:27017 ──▶  MongoDB Server (no seu computador)
                                                            └─ bancos → coleções → documentos
```

| Termo | O que é | Analogia |
|---|---|---|
| **MongoDB Server** (`mongod`) | O programa que guarda os dados e responde às consultas | A secretaria da escola |
| **localhost** | “Este mesmo computador” | O prédio onde você já está |
| **27017** | A porta em que o MongoDB atende | O número da sala da secretaria |
| **URI** | O endereço completo para conectar: `mongodb://localhost:27017` | O endereço + a sala, num papel só |

O MongoDB roda **em segundo plano**, como um serviço do Windows: você instala uma vez e ele fica disponível sempre que o computador estiver ligado.

---
## Etapa 1 · Abrir o terminal

No VS Code: menu **Terminal → New Terminal** (ou **Ctrl + '**). Um painel abre embaixo, com o **PowerShell**.

Todos os comandos das Etapas 2 a 4 são digitados **nesse terminal**, não nas células do notebook. Copie um comando por vez e tecle **Enter**.

---
## Etapa 2 · Conferir o `winget`

O `winget` é o instalador de programas pela linha de comando que já vem no Windows 10 e 11. No terminal:

```powershell
winget --version
```

Deve aparecer algo como `v1.9.25200`. Se aparecer *“winget não é reconhecido”*, instale ou atualize o **Instalador de Aplicativo** (*App Installer*) pela Microsoft Store e abra um terminal novo.

**Pronto quando:** o comando mostra um número de versão.

---
## Etapa 3 · Instalar o MongoDB Server

No terminal:

```powershell
winget install -e --id MongoDB.Server --accept-package-agreements --accept-source-agreements
```

- O Windows vai pedir permissão de administrador: clique em **Sim**.
- A instalação baixa o MongoDB Community Server e o registra como **serviço do Windows** chamado `MongoDB`, que já fica ligado.
- Leva alguns minutos. Espere aparecer *“Instalado com êxito”* (*Successfully installed*).

> **Sem permissão de administrador?** (computador do laboratório, por exemplo) Pule para a **Etapa 8 · Plano B**, que roda o MongoDB a partir de uma pasta, sem instalar.

**Pronto quando:** o `winget` termina com *Successfully installed*.

---
## Etapa 4 · Conferir se o MongoDB está rodando

No terminal:

```powershell
Get-Service MongoDB
```

O resultado deve mostrar **`Running`** na coluna *Status*.

| Se aparecer… | Faça |
|---|---|
| `Running` | Tudo certo, siga para a Etapa 5 |
| `Stopped` | Abra o terminal como administrador e rode `Start-Service MongoDB` |
| *Não foi possível localizar o serviço* | O serviço não foi criado. Use o **Plano B** da Etapa 8 |

**Pronto quando:** `Get-Service MongoDB` mostra `Running`.

---
## Etapa 5 · Criar o arquivo `.env`

Em vez de escrever o endereço do banco dentro de cada notebook, ele fica num arquivo separado chamado **`.env`**, que o Python lê com a biblioteca `python-dotenv`. Assim, se um dia o banco mudar de lugar (por exemplo, para a nuvem), você troca **uma linha** e todos os notebooks continuam funcionando.

Você pode criar o arquivo pelo terminal, **na pasta da disciplina**:

```powershell
Set-Content -Path .env -Value "MONGO_URI=mongodb://localhost:27017"
```

Ou pelo VS Code: no **Explorer**, **New File** → nome **`.env`** → escreva a linha abaixo → **Ctrl+S**.

```
MONGO_URI=mongodb://localhost:27017
```

**Pronto quando:** o arquivo `.env` aparece no Explorer, na mesma pasta deste notebook.

---
## Etapa 6 · Instalar as bibliotecas e testar a conexão

Agora sim, nas **células do notebook**. A primeira instala as bibliotecas no Python escolhido como kernel; a segunda lê o `.env` e testa a conexão com um `ping`.

In [1]:
%pip install -q pymongo python-dotenv pillow numpy matplotlib requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()                                   # lê o arquivo .env da pasta
client = MongoClient(os.environ["MONGO_URI"], serverSelectionTimeoutMS=5000)
client.admin.command("ping")
print("Conectado ao MongoDB!")
print("Versão do servidor:", client.server_info()["version"])

Conectado ao MongoDB!
Versão do servidor: 9.0.8


Se aparecer um erro, procure a mensagem na tabela da **Etapa 8**.

---
## Etapa 7 · Criar o primeiro banco

No MongoDB, bancos e coleções **nascem sozinhos** no momento em que o primeiro documento é inserido. Não existe `CREATE DATABASE`.

In [3]:
db = client["aula"]                 # banco "aula" (ainda não existe de verdade)
teste = db["teste"]                 # coleção "teste"

resultado = teste.insert_one({"mensagem": "Olá, MongoDB!", "autor": "seu nome aqui"})
print("Documento criado com _id:", resultado.inserted_id)

print("Bancos no seu servidor:", client.list_database_names())

Documento criado com _id: 6b04bd19acf92a0b42171961
Bancos no seu servidor: ['admin', 'aula', 'config', 'local']


Repare que, além do `aula`, aparecem os bancos `admin`, `config` e `local`: são bancos internos do próprio MongoDB.

---
## Etapa 8 · Problemas comuns

| Mensagem ou sintoma | Causa provável | Como resolver |
|---|---|---|
| `ServerSelectionTimeoutError` ou `connection refused` | O MongoDB não está rodando | Etapa 4: `Get-Service MongoDB`. Se não houver serviço, use o Plano B abaixo |
| `KeyError: 'MONGO_URI'` | O `.env` não foi encontrado ou a linha tem outro nome | O `.env` deve estar na mesma pasta do notebook, começando com `MONGO_URI=`. Depois, **Restart** no kernel |
| `ModuleNotFoundError: No module named 'pymongo'` | As bibliotecas foram instaladas em outro Python | Confira o kernel (**Select Kernel**, no canto superior direito) e rode a célula `%pip install` de novo |
| `winget` não é reconhecido | O Instalador de Aplicativo está ausente ou desatualizado | Atualize o **App Installer** pela Microsoft Store |
| O Windows pergunta sobre o firewall para o `mongod.exe` | Aviso normal na primeira execução | Pode permitir só em **redes privadas** |

### Plano B · Rodar sem instalar (sem administrador)

Use se o `winget` não puder instalar (falta de permissão) ou se o serviço `MongoDB` não existir.

1. Baixe o MongoDB Community Server em **https://www.mongodb.com/try/download/community**: *Platform* **Windows x64**, *Package* **zip**.
2. No terminal, descompacte e crie uma pasta para os dados (ajuste o nome do `.zip` se for diferente):
   ```powershell
   Expand-Archive "$HOME\Downloads\mongodb-windows-x86_64-*.zip" -DestinationPath "$HOME\mongodb"
   New-Item -ItemType Directory -Force "$HOME\mongodb-data"
   ```
3. Inicie o servidor:
   ```powershell
   $mongod = (Get-ChildItem "$HOME\mongodb" -Recurse -Filter mongod.exe | Select-Object -First 1).FullName
   & $mongod --dbpath "$HOME\mongodb-data"
   ```
4. **Deixe esse terminal aberto** enquanto usa os notebooks: fechá-lo desliga o banco. Para os comandos seguintes, abra outro terminal com o **+** do painel.

O endereço é o mesmo (`mongodb://localhost:27017`), então o `.env` e os notebooks não mudam.

### Checklist final
- [ ] MongoDB Server instalado (`winget`) ou rodando pelo Plano B
- [ ] `Get-Service MongoDB` mostra `Running` (ou o terminal do Plano B está aberto)
- [ ] Arquivo `.env` com `MONGO_URI=mongodb://localhost:27017`
- [ ] `Conectado ao MongoDB!` apareceu na Etapa 6
- [ ] Banco `aula` aparece na lista da Etapa 7

**Próximo:** `01_JSON_e_CRUD.ipynb`